# Pick and place across embodiments

The `Pick` and `Place` primitives are robot-agnostic: they consume a `Robot` (named groups, manipulators, IK) and a collision checker, so the *same* code picks and places a cube with different arms. Here we run it on the **Panda**, the **Kinova Gen3**, and one arm of the bimanual **Dexmate Vega**.

For each robot the scene is derived from its home end-effector pose: the cube is placed along the gripper's approach axis (so the pregrasp is the home pose), on a table, and the grasp uses the home orientation. Each plan is rendered as an inline animation; the held cube follows the gripper because a grasp is just a tree edge.

In [ ]:
import numpy as np
import pybullet as p
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML, display
from spatialmath import SE3

from prpl_kinematics.collision import PyBulletCollisionChecker
from prpl_kinematics.geometry.shapes import BoxShape
from prpl_kinematics.manipulation import Pick, Place
from prpl_kinematics.planning import BiRRTPlanner
from prpl_kinematics.robots import make_panda, make_kinova, make_vega
from prpl_kinematics.tree.joints import FixedJoint
from prpl_kinematics.tree.kinematic_tree import Edge, Node
from prpl_kinematics.tree.state import KinematicState
from prpl_kinematics.visualization import CameraParams, PyBulletRenderer, capture_image


def pick_and_place(make_robot, manipulator="arm", approach=0.12,
                   place_shift=(0.12, 0.0, 0.0), table_size=(0.4, 0.5, 0.02)):
    """Run Pick then Place with a home-derived cube/table scene; return (robot, plan, cube_pos)."""
    robot = make_robot()
    robot_links = set(robot.tree.nodes)  # everything not added below is the robot
    manip = robot.manipulators[manipulator]
    home_ee = robot.tree.forward_kinematics(manip.ee_frame, robot.home)
    rotation = np.asarray(home_ee.R)
    cube_pos = np.asarray(home_ee.t) + approach * rotation[:, 2]
    grasp = SE3.Rt(rotation, [0, 0, 0])
    place_shift = np.asarray(place_shift, dtype=float)
    placement = SE3(*(cube_pos + place_shift))
    table_center = cube_pos + place_shift / 2  # table spans both the pick and the place

    table = BoxShape(size=table_size)
    robot.tree.add_node(Node("table", visuals=[table], collisions=[table]))
    robot.tree.add_edge(Edge(robot.tree.root, "table", FixedJoint(
        name="tf", origin=SE3(table_center[0], table_center[1], cube_pos[2] - 0.055))))
    cube = BoxShape(size=(0.05, 0.05, 0.08))
    robot.tree.add_node(Node("cube", visuals=[cube], collisions=[cube]))
    robot.tree.add_edge(Edge(robot.tree.root, "cube", FixedJoint(name="cf", origin=SE3(*cube_pos))))

    checker = PyBulletCollisionChecker(p.connect(p.DIRECT))
    checker.load(robot.tree)
    checker.ignore(robot.allowed_collision_pairs)
    checker.ignore([("cube", "table")])  # the cube rests on the table
    checker.ignore([(link, "cube") for link in robot_links])  # the gripper grasps the cube
    planner = BiRRTPlanner(robot.groups[manip.group], checker.in_collision, np.random.default_rng(0), num_iters=2000)

    state = KinematicState.from_tree(robot.tree, robot.home)
    pick = Pick(robot, checker, planner, "cube", "table", [grasp],
                manipulator=manipulator, approach_distance=approach).plan(state)
    assert pick is not None, "pick failed"
    place = Place(robot, checker, planner, "cube", "table", [placement],
                  manipulator=manipulator, approach_distance=approach).plan(pick[-1])
    assert place is not None, "place failed"
    return robot, pick + place, cube_pos


def show(robot, plan, cube_pos, distance=1.1, yaw=55.0, pitch=-25.0):
    """Render a plan (applying each state's edges so the grasp follows) as an inline animation."""
    renderer = PyBulletRenderer(p.connect(p.DIRECT))
    renderer.load(robot.tree)
    camera = CameraParams(target=tuple(float(v) for v in cube_pos), distance=distance, yaw=yaw, pitch=pitch)
    images = []
    for plan_state in plan:
        renderer.render(plan_state.apply(robot.tree))
        images.append(capture_image(renderer.physics_client_id, camera))
    fig, ax = plt.subplots(figsize=(4, 3))
    ax.axis("off")
    canvas = ax.imshow(images[0])
    anim = animation.FuncAnimation(
        fig, lambda i: [canvas.set_data(images[i]) or canvas], frames=len(images), interval=40, blit=True)
    plt.close(fig)
    return HTML(anim.to_jshtml())

## Panda

In [ ]:
show(*pick_and_place(make_panda, approach=0.12, place_shift=(0.14, 0.0, 0.0)))

## Kinova Gen3

In [ ]:
show(*pick_and_place(make_kinova, approach=0.12, place_shift=(0.0, 0.16, 0.0)))

## Dexmate Vega (left arm)

Vega's stable home points the gripper down-forward, so the cube sits on a low table along that axis and the grasp is a short, smooth reach (the pregrasp is the home pose). We drive the `"left"` manipulator with a larger approach distance to suit the longer gripper.

In [ ]:
show(*pick_and_place(make_vega, manipulator="left", approach=0.20,
                     place_shift=(0.0, -0.14, 0.0), table_size=(0.28, 0.34, 0.02)),
     distance=1.7, yaw=135.0, pitch=-18.0)